# Experimentação

Este notebook orquestra a Fase 1 e 2 da etapa de experimentação, seguindo o protocolo descrito em `docs/experimentation.md`.

## Objetivos

## Setups e Imports

In [ ]:
import sys
from pathlib import Path

# Garante que a raiz do projeto está no sys.path
ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Manipulação de Dados
import pandas as pd
import numpy as np

# Visualização de Dados
import matplotlib.pyplot as plt
import seaborn as sns

# Pré-processamento e Modelagem
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate,
)


from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline


# Métricas de Avaliação
from sklearn.metrics import (
    roc_auc_score,
    recall_score,
    precision_score,
    f1_score,
    make_scorer
)


# Modelos de Machine Learning
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from src.models.mlp import MLP

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset


# MLflow
# ATENCAO: o servidor do MLflow precisa estar rodando antes de executar este notebook.
# Em um terminal separado, execute:
#   mlflow server --host 127.0.0.1 --port 5000
# Sem isso, as celulas de tracking falharao com ConnectionRefusedError.
import hashlib
import mlflow
import mlflow.sklearn

mlflow.set_tracking_uri("http://localhost:5000")

## Processamento dos Dados

In [ ]:
# dataload
df = pd.read_excel('../data/raw/Telco_customer_churn.xlsx')
df.head()

,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,...,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,...,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,...,Month-to-month,Yes,Electronic check,99.65,820.5,Yes,1,86,5372,Moved
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,...,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,...,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.3,Yes,1,89,5340,Competitor had better devices


In [ ]:
# Informações gerais sobre o dataset
print("=== INFORMAÇÕES GERAIS DO DATASET ===\n")
print(df.info())

# Shape
print("\n=== SHAPE DO DATASET ===")
print(f"Linhas: {df.shape[0]}, Colunas: {df.shape[1]}")

# Colunas
print("\n=== COLUNAS DO DATASET ===")
print(df.columns.tolist())

=== INFORMAÇÕES GERAIS DO DATASET ===

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 33 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   CustomerID         7043 non-null   object 
 1   Count              7043 non-null   int64  
 2   Country            7043 non-null   object 
 3   State              7043 non-null   object 
 4   City               7043 non-null   object 
 5   Zip Code           7043 non-null   int64  
 6   Lat Long           7043 non-null   object 
 7   Latitude           7043 non-null   float64
 8   Longitude          7043 non-null   float64
 9   Gender             7043 non-null   object 
 10  Senior Citizen     7043 non-null   object 
 11  Partner            7043 non-null   object 
 12  Dependents         7043 non-null   object 
 13  Tenure Months      7043 non-null   int64  
 14  Phone Service      7043 non-null   object 
 15  Multiple Lines     7043 non-null 

In [ ]:
# Transformação da coluna 'Total Charges' para numérica, tratando erros e preenchendo valores ausentes com 0.
df['Total Charges'] = pd.to_numeric(df['Total Charges'], errors='coerce')
df['Total Charges'] = df['Total Charges'].fillna(0)

In [ ]:
# dropando colunas irrelevantes para a modelagem
drop_cols = [
    'Country',
    'State',
    'Lat Long',
    'Churn Label',
    'Churn Reason',
    'Latitude',
    'Longitude',
    'City',
    'Churn Score',
    'Zip Code',
    'Count'
]

df.drop(columns=drop_cols, inplace=True)

In [ ]:
target = "Churn Value"
meta_cols = ["CLTV", "CustomerID"]

feature_cols = [
    col for col in df.columns
    if col not in [target] + meta_cols
]

X = df[feature_cols]
y = df[target]

metadata = df[meta_cols]


## Splits e Validação

In [ ]:
# Protocolo de validação cruzada
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


# 
X_train_val, X_test, y_train_val, y_test, metadata_train_val, metadata_test = train_test_split(
    X,
    y,
    metadata,
    test_size=0.3,
    stratify=y,
    random_state=42
)

# Checando os Splits
print("=== SPLITS ===")
print(f"Treino/Validação: {X_train_val.shape[0]} amostras")
print(f"Teste: {X_test.shape[0]} amostras")

# Checando a distribuição da variável alvo nos splits
print("\n=== DISTRIBUIÇÃO DA VARIÁVEL ALVO NOS SPLITS ===")
print("Treino/Validação:")
print(y_train_val.value_counts(normalize=True))
print("\nTeste:")
print(y_test.value_counts(normalize=True))

# Checando o formato dos dados
print("\n=== FORMATO DOS DADOS ===")
print(f"X_train_val: {X_train_val.shape}")
print(f"y_train_val: {y_train_val.shape}")
print(f"X_test: {X_test.shape}")
print(f"y_test: {y_test.shape}")


=== SPLITS ===
Treino/Validação: 4930 amostras
Teste: 2113 amostras

=== DISTRIBUIÇÃO DA VARIÁVEL ALVO NOS SPLITS ===
Treino/Validação:
Churn Value
0    0.734686
1    0.265314
Name: proportion, dtype: float64

Teste:
Churn Value
0    0.734501
1    0.265499
Name: proportion, dtype: float64

=== FORMATO DOS DADOS ===
X_train_val: (4930, 19)
y_train_val: (4930,)
X_test: (2113, 19)
y_test: (2113,)


## Baseline Inicial

- Treina Dummy e Regressão Logística e compara com a MLP e outros modelos de árvores.

In [ ]:
# Definindo a etapa de pré-processamento para variáveis categóricas com OHE e numéricas com passthrough
ohe = OneHotEncoder(handle_unknown="ignore")
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", ohe, make_column_selector(dtype_include=["object", "category"])),
        ("num", "passthrough", make_column_selector(dtype_exclude=["object", "category"])),
    ],
    remainder="drop",
)

# Dicionário de Pipelines para cada modelo (MLP será testada fora do Pipeline)
models = {
    "Dummy": Pipeline([
        ("prep", preprocessor),
        ("model", DummyClassifier(strategy="most_frequent")),
    ]),
    "LogisticRegression": Pipeline([
        ("prep", preprocessor),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
    ]),
    "DecisionTree": Pipeline([
        ("prep", preprocessor),
        ("model", DecisionTreeClassifier(random_state=42, class_weight="balanced")),
    ]),
    "RandomForest": Pipeline([
        ("prep", preprocessor),
        ("model", RandomForestClassifier(
            random_state=42, n_jobs=-1, class_weight="balanced_subsample"
        )),
    ]),
    "XGBoost": Pipeline([
        ("prep", preprocessor),
        ("model", XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            scale_pos_weight=(y_train_val == 0).sum() / (y_train_val == 1).sum(),
            random_state=42,
            n_jobs=-1,
        )),
    ])
}

In [ ]:
# definindo o scoring para avaliação dos modelos
scoring = {
    "pr_auc": "average_precision",
    "roc_auc": "roc_auc",
    "recall": make_scorer(recall_score, zero_division=0),
    "precision": make_scorer(precision_score, zero_division=0),
    "f1": make_scorer(f1_score, zero_division=0),
}

In [ ]:
metrics = ["pr_auc", "roc_auc", "recall", "precision", "f1"]
rows = []
fold_results = {}

for model_name, estimator in models.items():
    print(f"=== AVALIANDO MODELO: {model_name} ===")
    cv_res = cross_validate(
        estimator=estimator,
        X=X_train_val,
        y=y_train_val,
        cv=cv,
        scoring=scoring,
        n_jobs=1,
        return_train_score=False,
    )

    fold_results[model_name] = pd.DataFrame({
        "fold": np.arange(1, len(cv_res["fit_time"]) + 1),
        **{m: cv_res[f"test_{m}"] for m in metrics},
        "fit_time_s": cv_res["fit_time"],
        "score_time_s": cv_res["score_time"],
    })

    rows.append({
        "model": model_name,
        **{f"{m}_mean": cv_res[f"test_{m}"].mean() for m in metrics},
        "fit_time_mean_s": cv_res["fit_time"].mean(),
        "score_time_mean_s": cv_res["score_time"].mean(),
    })

results_cv = (
    pd.DataFrame(rows)
    .sort_values("pr_auc_mean", ascending=False)
    .reset_index(drop=True)
)


=== AVALIANDO MODELO: Dummy ===
=== AVALIANDO MODELO: LogisticRegression ===
=== AVALIANDO MODELO: DecisionTree ===
=== AVALIANDO MODELO: RandomForest ===
=== AVALIANDO MODELO: XGBoost ===


In [ ]:
import time
import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    recall_score,
    precision_score,
    f1_score,
)

from src.models.mlp import MLP, EarlyStopping

# ---------- Config ----------
MLP_EPOCHS = 80
MLP_BATCH_SIZE = 64
MLP_LR = 1e-3
MLP_WD = 1e-5
MLP_HIDDEN_DIM = 64
MLP_ACTIVATION = "relu"
MLP_THRESHOLD = 0.5

ES_PATIENCE = 8
ES_MIN_DELTA = 1e-4
ES_VAL_SIZE = 0.15

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def _rows(X, idx):
    return X.iloc[idx] if hasattr(X, "iloc") else X[idx]


def _to_dense_float32(x):
    if sp.issparse(x):
        x = x.toarray()
    return np.asarray(x, dtype=np.float32)


mlp_folds = []

for fold, (tr_idx, va_idx) in enumerate(cv.split(X_train_val, y_train_val), start=1):
    fit_start = time.perf_counter()

    X_tr_raw = _rows(X_train_val, tr_idx)
    X_va_raw = _rows(X_train_val, va_idx)
    y_tr = np.asarray(_rows(y_train_val, tr_idx), dtype=np.float32)
    y_va = np.asarray(_rows(y_train_val, va_idx), dtype=np.float32)

    prep_fold = clone(preprocessor)
    X_tr_enc = prep_fold.fit_transform(X_tr_raw, y_tr)
    X_va_enc = prep_fold.transform(X_va_raw)

    idx_all = np.arange(len(y_tr))
    idx_tr, idx_es = train_test_split(
        idx_all,
        test_size=ES_VAL_SIZE,
        stratify=y_tr,
        random_state=42 + fold,
    )

    X_tr_fit = X_tr_enc[idx_tr]
    y_tr_fit = y_tr[idx_tr]
    X_tr_es = X_tr_enc[idx_es]
    y_tr_es = y_tr[idx_es]

    scaler = StandardScaler(with_mean=False)
    X_tr_fit = scaler.fit_transform(X_tr_fit)
    X_tr_es = scaler.transform(X_tr_es)
    X_va_sc = scaler.transform(X_va_enc)

    X_tr_fit = _to_dense_float32(X_tr_fit)
    X_tr_es = _to_dense_float32(X_tr_es)
    X_va_sc = _to_dense_float32(X_va_sc)

    train_loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(
            torch.tensor(X_tr_fit, dtype=torch.float32),
            torch.tensor(y_tr_fit, dtype=torch.float32),
        ),
        batch_size=MLP_BATCH_SIZE,
        shuffle=True,
    )
    es_loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(
            torch.tensor(X_tr_es, dtype=torch.float32),
            torch.tensor(y_tr_es, dtype=torch.float32),
        ),
        batch_size=MLP_BATCH_SIZE,
        shuffle=False,
    )

    model = MLP(
        input_dim=X_tr_fit.shape[1],
        hidden_dim=MLP_HIDDEN_DIM,
        output_dim=1,
        activation=MLP_ACTIVATION,
    ).to(DEVICE)

    pos = float((y_tr_fit == 1).sum())
    neg = float((y_tr_fit == 0).sum())
    pos_weight = torch.tensor([neg / max(pos, 1.0)], dtype=torch.float32).to(DEVICE)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = optim.Adam(model.parameters(), lr=MLP_LR, weight_decay=MLP_WD)
    early_stopper = EarlyStopping(patience=ES_PATIENCE, min_delta=ES_MIN_DELTA)

    epochs_trained = 0
    for epoch in range(1, MLP_EPOCHS + 1):
        model.train()
        for xb, yb in train_loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE).unsqueeze(1)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

        model.eval()
        es_losses = []
        with torch.no_grad():
            for xb, yb in es_loader:
                xb = xb.to(DEVICE)
                yb = yb.to(DEVICE).unsqueeze(1)
                logits = model(xb)
                es_losses.append(criterion(logits, yb).item())

        mean_es_loss = float(np.mean(es_losses))
        epochs_trained = epoch

        if early_stopper.step(mean_es_loss, model):
            break

    early_stopper.restore_best_weights(model)
    fit_time_s = time.perf_counter() - fit_start

    score_start = time.perf_counter()
    X_va_t = torch.tensor(X_va_sc, dtype=torch.float32).to(DEVICE)

    model.eval()
    with torch.no_grad():
        logits = model(X_va_t).squeeze(1)
        prob = torch.sigmoid(logits).cpu().numpy()

    pred = (prob >= MLP_THRESHOLD).astype(int)

    mlp_folds.append(
        {
            "fold": fold,
            "epochs_trained": epochs_trained,
            "best_es_loss": early_stopper.best_loss,
            "pr_auc": average_precision_score(y_va, prob),
            "roc_auc": roc_auc_score(y_va, prob),
            "recall": recall_score(y_va, pred, zero_division=0),
            "precision": precision_score(y_va, pred, zero_division=0),
            "f1": f1_score(y_va, pred, zero_division=0),
            "fit_time_s": fit_time_s,
            "score_time_s": time.perf_counter() - score_start,
        }
    )

fold_results["MLP"] = pd.DataFrame(mlp_folds)

mlp_df = fold_results["MLP"]
mlp_row = {
    "model": "MLP",
    "pr_auc_mean": mlp_df["pr_auc"].mean(),
    "roc_auc_mean": mlp_df["roc_auc"].mean(),
    "recall_mean": mlp_df["recall"].mean(),
    "precision_mean": mlp_df["precision"].mean(),
    "f1_mean": mlp_df["f1"].mean(),
    "fit_time_mean_s": mlp_df["fit_time_s"].mean(),
    "score_time_mean_s": mlp_df["score_time_s"].mean(),
}

results_cv = (
    pd.concat(
        [results_cv[results_cv["model"] != "MLP"], pd.DataFrame([mlp_row])],
        ignore_index=True,
    )
    .sort_values("pr_auc_mean", ascending=False)
    .reset_index(drop=True)
)


# display(fold_results["MLP"].round(4))

display(results_cv.round(4))



,fold,epochs_trained,best_es_loss,pr_auc,roc_auc,recall,precision,f1,fit_time_s,score_time_s
0,1,25,0.5903,0.6586,0.8437,0.7586,0.5224,0.6188,1.8685,0.0050
1,2,35,0.6356,0.6556,0.8632,0.8008,0.5529,0.6541,1.3143,0.0047
2,3,12,0.7656,0.6305,0.8402,0.8092,0.4965,0.6154,0.4416,0.0041
3,4,21,0.6891,0.6840,0.8585,0.8359,0.5202,0.6413,0.7481,0.0044
4,5,21,0.7726,0.6913,0.8633,0.8130,0.5392,0.6484,0.7500,0.0043


,model,pr_auc_mean,roc_auc_mean,recall_mean,precision_mean,f1_mean,fit_time_mean_s,score_time_mean_s
0,LogisticRegression,0.6782,0.8576,0.8134,0.5314,0.6427,0.0203,0.0100
1,MLP,0.6640,0.8538,0.8035,0.5263,0.6356,1.0245,0.0045
2,XGBoost,0.6492,0.8438,0.6743,0.5685,0.6167,0.0605,0.0148
3,RandomForest,0.6323,0.8393,0.5091,0.6467,0.5692,0.1671,0.0588
4,DecisionTree,0.3927,0.6684,0.5099,0.5147,0.5120,0.0240,0.0115
5,Dummy,0.2653,0.5000,0.0000,0.0000,0.0000,0.0090,0.0109


### Logging no MLflow

## Feature Engineering

### Obejtivo

Adicionar poder preditivo aos modelos de forma controlada

In [ ]:
# Adicionando as Features


